# NyaayKhel — 02: Dataset Builder

**Purpose:** Run YOLOv8-pose over all labeled clips and build the numpy keypoint-sequence
dataset that feeds the GRU/TCN classifier in notebook 03.

**Prerequisites:**
- `data/raw/clip_index.csv` exists with `label` column filled in (from CVAT/Label Studio)
- `model/yolov8n_pose.tflite` exists (exported in notebook 01, or use `.pt` directly here)

**What this notebook produces:**
- `data/processed/windows_X.npy` — shape `(N_windows, 30, 102)` — input features
- `data/processed/labels_y.npy` — shape `(N_windows,)` — integer class labels
- `data/processed/label_map.json` — class name → integer mapping
- `data/processed/dataset_stats.json` — per-class counts, split sizes, clip metadata

**Exit gate:** numpy arrays saved, per-class counts logged, ≥ 120 windows total.

## Cell 1: Install & Imports

In [ ]:
!pip install -q ultralytics opencv-python-headless

import os, csv, json, time
import numpy as np
import cv2
from pathlib import Path
from collections import Counter, defaultdict
from ultralytics import YOLO
import torch

print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

## Cell 2: Paths & Config

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/NyaayKhel'
else:
    BASE_DIR = '/content/NyaayKhel'

RAW_DIR       = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODEL_DIR     = os.path.join(BASE_DIR, 'model')
os.makedirs(PROCESSED_DIR, exist_ok=True)

CLIP_INDEX_PATH = os.path.join(RAW_DIR, 'clip_index.csv')

# ── MODEL CONFIG ──────────────────────────────────────────────────────────────
# Use the .pt model directly on Colab GPU (faster than TFLite here)
POSE_MODEL_NAME = 'yolov8n-pose.pt'
PERSON_CONF     = 0.5    # min person detection confidence
KP_CONF_THRESH  = 0.3    # keypoints below this are zeroed out

# ── WINDOW CONFIG ────────────────────────────────────────────────────────────
# Must match the Android app's sliding window settings.
WINDOW_SIZE     = 30     # frames per classification window
WINDOW_STRIDE   = 10     # stride between windows (overlap = 20 frames)
TARGET_FPS      = 10.0   # must match Android MainViewModel.sampleFps
MAX_PERSONS     = 2      # two scene-level persons, sorted left-to-right
N_KEYPOINTS     = 17     # YOLOv8-pose outputs
# Feature vector size per frame: MAX_PERSONS * N_KEYPOINTS * 3 (x, y, conf)
FEATURE_DIM     = MAX_PERSONS * N_KEYPOINTS * 3  # = 102

# ── LABEL MAP ────────────────────────────────────────────────────────────────
# Integer encoding for the 4 classes.
# Must be consistent with notebook 03 and the Android app's class index.
LABEL_MAP = {
    'raid_start':    0,
    'touch':         1,
    'escape_return': 2,
    'neutral':       3,
}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

print(f'Base dir:       {BASE_DIR}')
print(f'Processed dir:  {PROCESSED_DIR}')
print(f'Clip index:     {CLIP_INDEX_PATH}')
print(f'Window size:    {WINDOW_SIZE} frames | Stride: {WINDOW_STRIDE}')
print(f'Target FPS:     {TARGET_FPS} (30 frames = {WINDOW_SIZE / TARGET_FPS:.1f}s)')
print(f'Feature dim:    {FEATURE_DIM} ({MAX_PERSONS} persons x {N_KEYPOINTS} kp x 3)')
print(f'Label map:      {LABEL_MAP}')


## Cell 3: Load Clip Index — Filter Labeled Clips Only

In [ ]:
if not os.path.exists(CLIP_INDEX_PATH):
    raise FileNotFoundError(
        f'clip_index.csv not found at {CLIP_INDEX_PATH}\n'
        'Run 00_data_collection.ipynb first, then label clips in CVAT/Label Studio.'
    )

with open(CLIP_INDEX_PATH, 'r', encoding='utf-8') as f:
    all_clips = list(csv.DictReader(f))

# Filter: keep only clips with a valid label AND that exist on disk
labeled_clips = [
    c for c in all_clips
    if c.get('label', '').strip() in LABEL_MAP
    and os.path.exists(c.get('file', ''))
]

unlabeled = [c for c in all_clips if c.get('label', '').strip() not in LABEL_MAP]
missing   = [c for c in all_clips if c.get('label', '').strip() in LABEL_MAP
             and not os.path.exists(c.get('file', ''))]

print(f'Total clips in index: {len(all_clips)}')
print(f'Labeled + on disk:    {len(labeled_clips)}')
print(f'Unlabeled (skipped):  {len(unlabeled)}')
print(f'Missing files:        {len(missing)}')
print()

label_counts = Counter(c['label'] for c in labeled_clips)
print('Label distribution:')
for label, count in sorted(label_counts.items()):
    print(f'  {label:20s}: {count:4d} clips')

if len(labeled_clips) < 100:
    print()
    print('WARNING: Fewer than 100 labeled clips. Model quality may suffer.')
    print('Consider labeling more clips before training.')

## Cell 4: Load Pose Model

In [ ]:
model = YOLO(POSE_MODEL_NAME)
print(f'Pose model loaded: {POSE_MODEL_NAME}')
print(f'Device: {next(model.model.parameters()).device}')

## Cell 5: Keypoint Extraction Functions

In [ ]:
def extract_keypoints_from_frame(frame, model, person_conf, kp_conf_thresh,
                                  max_persons, n_keypoints):
    """
    Run YOLOv8-pose on a single frame.
    Returns array of shape (max_persons, n_keypoints, 3) - x, y, conf.
    Persons sorted by x-position (left to right); padded with zeros if < max_persons.
    Keypoints with conf < kp_conf_thresh are zeroed.
    """
    results = model(frame, conf=person_conf, verbose=False)
    result = results[0]

    out = np.zeros((max_persons, n_keypoints, 3), dtype=np.float32)

    if result.keypoints is None or len(result.boxes) == 0:
        return out

    kp_data = result.keypoints.data.cpu().numpy()   # (N_det, 17, 3)
    boxes   = result.boxes.xyxy.cpu().numpy()       # (N_det, 4)

    # Sort detections by box center-x (left person first)
    center_x = (boxes[:, 0] + boxes[:, 2]) / 2
    order = np.argsort(center_x)
    kp_data = kp_data[order]

    # Take up to max_persons
    n_take = min(len(kp_data), max_persons)
    taken = kp_data[:n_take].copy()

    # Zero low-confidence keypoints
    mask = taken[:, :, 2] < kp_conf_thresh
    taken[mask] = 0

    out[:n_take] = taken
    return out


def normalize_frame_keypoints(kp_array, frame_w, frame_h):
    """
    Normalise x,y coordinates to [0, 1] relative to frame dimensions.
    Confidence values are kept as-is.
    kp_array: (max_persons, n_keypoints, 3)
    """
    normed = kp_array.copy()
    normed[:, :, 0] /= frame_w  # x
    normed[:, :, 1] /= frame_h  # y
    np.clip(normed[:, :, :2], 0.0, 1.0, out=normed[:, :, :2])
    return normed


def clip_to_windows(clip_path, label_int, model, config):
    """
    Process a single clip:
      1. Sample frames at config['target_fps'] (matching Android video mode)
      2. Extract per-frame keypoints
      3. Normalise coordinates
      4. Slide a window of size WINDOW_SIZE with stride WINDOW_STRIDE
      5. Return list of (window_array, label_int) tuples

    window_array shape: (WINDOW_SIZE, FEATURE_DIM), ready for GRU input.
    """
    cap = cv2.VideoCapture(clip_path)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    native_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    if native_fps <= 0:
        native_fps = 30.0
    frame_stride = max(1, round(native_fps / config['target_fps']))

    frame_kps = []
    frame_num = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_num % frame_stride == 0:
            kps = extract_keypoints_from_frame(
                frame, model,
                config['person_conf'], config['kp_conf_thresh'],
                config['max_persons'], config['n_keypoints']
            )
            normed = normalize_frame_keypoints(kps, w, h)
            frame_kps.append(normed.flatten())  # (FEATURE_DIM,)
        frame_num += 1
    cap.release()

    if len(frame_kps) < config['window_size']:
        return []

    windows = []
    for start in range(0, len(frame_kps) - config['window_size'] + 1, config['window_stride']):
        window = np.stack(frame_kps[start : start + config['window_size']])
        windows.append((window, label_int))

    return windows


print('Extraction functions defined.')
print(f'Temporal contract: target_fps={TARGET_FPS}, window={WINDOW_SIZE / TARGET_FPS:.1f}s, stride={WINDOW_STRIDE / TARGET_FPS:.1f}s')


## Cell 6: Build the Dataset (Main Loop)

In [ ]:
config = {
    'person_conf':  PERSON_CONF,
    'kp_conf_thresh': KP_CONF_THRESH,
    'max_persons':  MAX_PERSONS,
    'n_keypoints':  N_KEYPOINTS,
    'window_size':  WINDOW_SIZE,
    'window_stride': WINDOW_STRIDE,
    'target_fps': TARGET_FPS,
    'feature_dim':  FEATURE_DIM,
}

all_windows = []   # list of (window_array, label_int, source_video_id)
per_class_counts = defaultdict(int)
skipped_clips = []
t_start = time.time()

print(f'Processing {len(labeled_clips)} labeled clips...')
print()

for i, clip in enumerate(labeled_clips):
    label_str = clip['label'].strip()
    label_int = LABEL_MAP[label_str]
    clip_path = clip['file']
    source_video_id = clip.get('source_video_id') or clip['clip_id'].split('_t')[0]

    try:
        windows = clip_to_windows(clip_path, label_int, model, config)
        if not windows:
            skipped_clips.append({'clip': clip['clip_id'], 'reason': 'too short'})
            continue
        all_windows.extend((w, l, source_video_id) for (w, l) in windows)
        per_class_counts[label_str] += len(windows)

        if (i + 1) % 20 == 0 or (i + 1) == len(labeled_clips):
            elapsed = time.time() - t_start
            print(f'  [{i+1:4d}/{len(labeled_clips)}] '
                  f'{label_str:20s} | windows this clip: {len(windows):3d} '
                  f'| total: {len(all_windows):5d} | elapsed: {elapsed:.0f}s')
    except Exception as e:
        skipped_clips.append({'clip': clip['clip_id'], 'reason': str(e)[:100]})
        print(f'  SKIP {clip["clip_id"]}: {e}')

print()
print(f'Total windows built: {len(all_windows)}')
print(f'Clips skipped:       {len(skipped_clips)}')
print(f'Source videos seen:  {len(set(v for _, _, v in all_windows))}')
print()
print('Windows per class:')
for label, count in sorted(per_class_counts.items()):
    print(f'  {label:20s}: {count:5d}')


## Cell 7: Shuffle & Train/Test Split

In [ ]:
import random
from collections import defaultdict, Counter

RANDOM_SEED = 42
TEST_VIDEO_FRACTION = 0.2

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

windows_by_video = defaultdict(list)
for window, label, source_video_id in all_windows:
    windows_by_video[source_video_id].append((window, label))

all_video_ids = sorted(windows_by_video.keys())
random.shuffle(all_video_ids)

if len(all_video_ids) < 3:
    raise ValueError('Need >=3 distinct source videos for train/test/demo split.')

demo_video_id = all_video_ids[0]
remaining_video_ids = all_video_ids[1:]

n_test_videos = max(1, int(len(remaining_video_ids) * TEST_VIDEO_FRACTION))
test_video_ids  = set(remaining_video_ids[:n_test_videos])
train_video_ids = set(remaining_video_ids[n_test_videos:])

def _collect(video_ids):
    return [wl for vid in video_ids for wl in windows_by_video[vid]]

train_windows = _collect(train_video_ids)
test_windows  = _collect(test_video_ids)
random.shuffle(train_windows)
random.shuffle(test_windows)

split_sets = [train_video_ids, test_video_ids, {demo_video_id}]
assert train_video_ids.isdisjoint(test_video_ids)
assert train_video_ids.isdisjoint({demo_video_id})
assert test_video_ids.isdisjoint({demo_video_id})

print(f'Videos - train: {len(train_video_ids)} | test: {len(test_video_ids)} | demo: {demo_video_id}')
print(f'Total windows:  {len(all_windows)}')
print(f'Train windows:  {len(train_windows)}')
print(f'Test windows:   {len(test_windows)}')
print()

train_counts = Counter(label for _, label in train_windows)
test_counts  = Counter(label for _, label in test_windows)
print(f'{"Class":20s}  {"Train":>8}  {"Test":>8}')
print('-' * 42)
for li in sorted(LABEL_MAP.values()):
    name = INV_LABEL_MAP[li]
    print(f'{name:20s}  {train_counts.get(li,0):>8}  {test_counts.get(li,0):>8}')

print()
print('No source_video_id appears in more than one split.')


## Cell 8: Save Numpy Arrays

In [ ]:
def unzip_windows(window_list):
    X = np.stack([w for w, _ in window_list]).astype(np.float32)
    y = np.array([l for _, l in window_list], dtype=np.int64)
    return X, y

X_train, y_train = unzip_windows(train_windows)
X_test,  y_test  = unzip_windows(test_windows)

print(f'X_train shape: {X_train.shape}  (windows, timesteps, features)')
print(f'y_train shape: {y_train.shape}')
print(f'X_test  shape: {X_test.shape}')
print(f'y_test  shape: {y_test.shape}')

# Save arrays
np.save(os.path.join(PROCESSED_DIR, 'X_train.npy'), X_train)
np.save(os.path.join(PROCESSED_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(PROCESSED_DIR, 'X_test.npy'),  X_test)
np.save(os.path.join(PROCESSED_DIR, 'y_test.npy'),  y_test)

# Save label map
with open(os.path.join(PROCESSED_DIR, 'label_map.json'), 'w') as f:
    json.dump(LABEL_MAP, f, indent=2)

# Save dataset stats
stats = {
    'window_size': WINDOW_SIZE,
    'window_stride': WINDOW_STRIDE,
    'target_fps': TARGET_FPS,
    'feature_dim': FEATURE_DIM,
    'max_persons': MAX_PERSONS,
    'n_keypoints': N_KEYPOINTS,
    'label_map': LABEL_MAP,
    'total_clips_labeled': len(labeled_clips),
    'clips_skipped': len(skipped_clips),
    'total_windows': len(all_windows),
    'train_windows': len(train_windows),
    'test_windows':  len(test_windows),
    'demo_video_id': demo_video_id,
    'train_video_ids': sorted(train_video_ids),
    'test_video_ids': sorted(test_video_ids),
    'train_class_counts': {INV_LABEL_MAP[k]: int(v) for k, v in train_counts.items()},
    'test_class_counts':  {INV_LABEL_MAP[k]: int(v) for k, v in test_counts.items()},
    'random_seed': RANDOM_SEED,
    'test_video_fraction': TEST_VIDEO_FRACTION,
    'pose_model': POSE_MODEL_NAME,
    'person_conf': PERSON_CONF,
    'kp_conf_thresh': KP_CONF_THRESH,
}
with open(os.path.join(PROCESSED_DIR, 'dataset_stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)

print()
print('Saved to:', PROCESSED_DIR)
for fname in ['X_train.npy', 'y_train.npy', 'X_test.npy', 'y_test.npy',
               'label_map.json', 'dataset_stats.json']:
    fpath = os.path.join(PROCESSED_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {fname:30s}  {size_kb:.1f} KB')


## Cell 9: Exit Gate

In [ ]:
checks = [
    ('X_train.npy saved',        os.path.exists(os.path.join(PROCESSED_DIR, 'X_train.npy'))),
    ('X_test.npy saved',         os.path.exists(os.path.join(PROCESSED_DIR, 'X_test.npy'))),
    ('label_map.json saved',     os.path.exists(os.path.join(PROCESSED_DIR, 'label_map.json'))),
    ('dataset_stats.json saved', os.path.exists(os.path.join(PROCESSED_DIR, 'dataset_stats.json'))),
    ('Total windows >= 120',     len(all_windows) >= 120),
    ('All 4 classes present',    len(per_class_counts) == 4),
    ('Test set >= 20 windows',   len(test_windows) >= 20),
    ('X_train shape correct',    X_train.ndim == 3 and X_train.shape[1] == WINDOW_SIZE and X_train.shape[2] == FEATURE_DIM),
    ('No source-video leakage',  train_video_ids.isdisjoint(test_video_ids) and demo_video_id not in train_video_ids and demo_video_id not in test_video_ids),
]

print('=' * 55)
print('PHASE B DATASET EXIT GATE')
print('=' * 55)
all_pass = True
for label, passed in checks:
    status = 'PASS' if passed else 'FAIL'
    if not passed:
        all_pass = False
    print(f'  [{status}]  {label}')

print()
if all_pass:
    print('ALL CHECKS PASSED. Proceed to 03_train_classifier.ipynb.')
else:
    print('SOME CHECKS FAILED. Fix before training.')
    if len(all_windows) < 120:
        print('  -> Label more clips in CVAT/Label Studio.')
    if len(per_class_counts) < 4:
        missing_cls = set(LABEL_MAP.keys()) - set(per_class_counts.keys())
        print(f'  -> Missing clips for classes: {missing_cls}')
